# OSINT-Bench Case Browser

This notebook loads the JSON case files and renders them in a more readable review format.

Run the cells from top to bottom, then change `CASE_INDEX` or `CASE_FILENAME` to inspect a specific case.

In [1]:
from pathlib import Path
import json
from html import escape
from IPython.display import HTML, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "cases").exists():
    candidates = [Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "cases").exists():
            ROOT = candidate
            break

CASES_DIR = ROOT / "cases"
case_files = sorted(CASES_DIR.glob("*.json"))

if not case_files:
    raise FileNotFoundError(f"No case JSON files found in {CASES_DIR}")

cases = []
for path in case_files:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    data["__file__"] = path.name
    cases.append(data)

len(cases)

10

In [2]:
def html_table(rows, headers):
    parts = [
        "<table style='border-collapse:collapse;width:100%;font-size:14px'>",
        "<thead><tr>",
    ]
    for header in headers:
        parts.append(
            f"<th style='text-align:left;border-bottom:2px solid #999;padding:8px'>{escape(str(header))}</th>"
        )
    parts.append("</tr></thead><tbody>")

    for row in rows:
        parts.append("<tr>")
        for cell in row:
            parts.append(
                f"<td style='vertical-align:top;border-bottom:1px solid #ddd;padding:8px'>{cell}</td>"
            )
        parts.append("</tr>")

    parts.append("</tbody></table>")
    return HTML("".join(parts))


def as_list(items):
    if not items:
        return ""
    return "<ul style='margin:0;padding-left:18px'>" + "".join(
        f"<li>{escape(str(item))}</li>" for item in items
    ) + "</ul>"


def source_link(source):
    url = source.get("uri", "")
    title = source.get("content", {}).get("page_title", url)
    return f"<a href='{escape(url)}' target='_blank'>{escape(title)}</a>"


def render_case_index(case_list):
    rows = []
    for i, case in enumerate(case_list, start=1):
        gt = case.get("ground_truth", {})
        rows.append([
            str(i),
            escape(case.get("__file__", "")),
            escape(case.get("case_id", "")),
            escape(case.get("difficulty", "")),
            escape(case.get("investigation_category", "")),
            escape(case.get("target", "")),
            str(len(case.get("sources", []))),
            str(len(gt.get("entities", []))),
            str(len(gt.get("relations", []))),
        ])
    display(Markdown("## Case Index"))
    display(html_table(rows, [
        "#", "File", "Case ID", "Difficulty", "Category", "Target", "Sources", "Entities", "Relations"
    ]))


def render_case(case):
    gt = case.get("ground_truth", {})
    display(Markdown(f"# {case.get('case_id', 'Unknown Case')}"))
    display(Markdown(
        f"**File:** `{case.get('__file__', '')}`  \n"
        f"**Target:** {case.get('target', '')}  \n"
        f"**Category:** {case.get('investigation_category', '')}  \n"
        f"**Difficulty:** {case.get('difficulty', '')}  \n\n"
        f"**Investigation Goal:** {case.get('investigation_goal', '')}"
    ))

    if gt.get("ground_truth_summary"):
        display(Markdown("## Ground Truth Summary"))
        display(Markdown(gt["ground_truth_summary"]))

    display(Markdown("## Expected Findings"))
    finding_rows = []
    for finding in gt.get("key_findings", []):
        finding_rows.append([
            escape(finding.get("question", "")),
            escape(finding.get("answer", "")),
            as_list(finding.get("supporting_entities", [])),
        ])
    display(html_table(finding_rows, ["Question", "Answer", "Supporting Entities"]))

    display(Markdown("## Sources"))
    source_rows = []
    for source in case.get("sources", []):
        content = source.get("content", {})
        source_rows.append([
            escape(source.get("source_id", "")),
            source_link(source),
            escape(source.get("type", "")),
            escape(content.get("publication_date", "")),
            escape(content.get("date_accessed", "")),
            escape(content.get("evidence_summary", "")),
            as_list(content.get("supports_findings", [])),
        ])
    display(html_table(source_rows, [
        "ID", "Page", "Type", "Published", "Accessed", "Evidence Summary", "Supports"
    ]))

    display(Markdown("## Entities"))
    entity_rows = []
    for entity in gt.get("entities", []):
        entity_rows.append([
            escape(entity.get("id", "")),
            escape(entity.get("type", "")),
            escape(entity.get("value", "")),
            as_list(entity.get("source_references", [])),
        ])
    display(html_table(entity_rows, ["ID", "Type", "Value", "Source References"]))

    display(Markdown("## Relationships"))
    relation_rows = []
    for relation in gt.get("relations", []):
        relation_rows.append([
            escape(relation.get("source", "")),
            escape(relation.get("relationship_type", "")),
            escape(relation.get("target", "")),
            as_list(relation.get("source_references", [])),
        ])
    display(html_table(relation_rows, ["Source", "Relationship", "Target", "Evidence Sources"]))

    contradictions = gt.get("contradictions", [])
    if contradictions:
        display(Markdown("## Contradictions"))
        contradiction_rows = []
        for item in contradictions:
            contradiction_rows.append([
                escape(item.get("contradiction_id", "")),
                escape(item.get("description", "")),
                escape(item.get("reconciliation", "")),
                as_list(item.get("source_references", [])),
            ])
        display(html_table(contradiction_rows, [
            "ID", "Description", "Reconciliation", "Source References"
        ]))

    reasoning = gt.get("reasoning_proof_chains", [])
    if reasoning:
        display(Markdown("## Reasoning Chain"))
        reasoning_rows = []
        for step in reasoning:
            reasoning_rows.append([
                str(step.get("step_index", "")),
                as_list(step.get("premise_entities", [])),
                as_list(step.get("premise_relations", [])),
                escape(step.get("conclusion", "")),
            ])
        display(html_table(reasoning_rows, [
            "Step", "Premise Entities", "Premise Relations", "Conclusion"
        ]))

    if case.get("evaluation_notes"):
        display(Markdown("## Evaluation Notes"))
        display(Markdown(case["evaluation_notes"]))


In [3]:
render_case_index(cases)

## Case Index

#,File,Case ID,Difficulty,Category,Target,Sources,Entities,Relations
1,case_001_3cx_cascading_supply_chain.json,OSINT-001,Hard,Threat Actor & Infrastructure Attribution,3CX DesktopApp 2023 software supply chain compromise,9,25,19
2,case_002_candidate_conti_ransomware_leaks.json,OSINT-CAND-002,Hard,Threat Actor & Infrastructure Attribution,Conti ransomware leaks and public attribution record,4,7,6
3,case_003_candidate_lazarus_ronin_harmony.json,OSINT-CAND-003,Hard,Cryptocurrency & Asset Tracing,Lazarus-linked Ronin Bridge and Harmony Horizon cryptocurrency thefts,4,7,6
4,case_004_candidate_guccifer_dcleaks.json,OSINT-CAND-004,Hard,Social Media & Sockpuppet Networks,Guccifer 2.0 and DCLeaks attribution chain,2,8,6
5,case_005_candidate_onecoin_cryptoqueen.json,OSINT-CAND-005,Hard,Cryptocurrency & Asset Tracing,OneCoin / Cryptoqueen network,3,8,6
6,case_006_candidate_moveit_clop.json,OSINT-CAND-006,Hard,Threat Actor & Infrastructure Attribution,MOVEit Transfer / CL0P mass exploitation,2,8,5
7,case_007_candidate_hyperverse_hyperfund.json,OSINT-CAND-007,Hard,Cryptocurrency & Asset Tracing,HyperFund / HyperVerse alleged crypto fraud network,4,7,6
8,case_008_candidate_wirecard_marsalek.json,OSINT-CAND-008,Hard,Shell Company & Corporate Fraud,Wirecard / Jan Marsalek public fraud trail,2,8,7
9,case_009_candidate_pandora_jordan_property.json,OSINT-CAND-009,Hard,Shell Company & Corporate Fraud,Pandora Papers - King Abdullah II property network,4,10,9
10,case_010_candidate_ira_influence_operation.json,OSINT-CAND-010,Hard,Social Media & Sockpuppet Networks,Internet Research Agency 2016 influence operation,4,12,11


In [4]:
CASE_INDEX = 1
CASE_FILENAME = None

selected_case = None
if CASE_FILENAME:
    selected_case = next(case for case in cases if case["__file__"] == CASE_FILENAME)
else:
    selected_case = cases[CASE_INDEX - 1]

render_case(selected_case)

# OSINT-001

**File:** `case_001_3cx_cascading_supply_chain.json`  
**Target:** 3CX DesktopApp 2023 software supply chain compromise  
**Category:** Threat Actor & Infrastructure Attribution  
**Difficulty:** Hard  

**Investigation Goal:** Reconstruct the public evidence chain showing how the 2023 3CX DesktopApp compromise was initiated, how it propagated into 3CX's build environment, and which threat activity cluster it was publicly attributed to.

## Ground Truth Summary

The 2023 3CX DesktopApp incident is a strong OSINT benchmark case because the correct answer requires linking multiple public reports across time. The surface event was a trojanized 3CX DesktopApp affecting Windows and macOS Electron builds. Later 3CX and Mandiant reporting identified the root cause as a prior compromise of Trading Technologies' X_TRADER installer, which a 3CX employee installed on a personal computer in 2022. That installer deployed VEILEDSIGNAL, enabling theft of corporate credentials and access into the 3CX environment. The actor then moved laterally, compromised Windows and macOS build environments, and produced trojanized 3CX DesktopApp builds that used staged malware including SUDDENICON and ICONICSTEALER. Attribution should be stated carefully: Mandiant tracked the activity as UNC4736 and assessed a North Korean nexus, while Kaspersky and MITRE provide supporting context around Lazarus/AppleJeus-style cryptocurrency-focused activity. A correct investigation must reconcile early incomplete vendor statements with later root-cause findings, and distinguish broad installer exposure from selective follow-on compromise.

## Expected Findings

...

ID,Description,Reconciliation,Source References
CONTRA-001,"The initial 3CX alert described the visible issue as appearing to involve a bundled library compiled into the Electron app via Git, whereas later 3CX/Mandiant reporting identified the upstream intrusion vector as a trojanized Trading Technologies X_TRADER installer that compromised a 3CX employee system and led into the corporate build environment.","Treat the March 30 vendor alert as an early, incomplete incident notice about the compromised shipped app and the April 20 3CX/Mandiant publications as the later root-cause finding. The correct investigation conclusion is that the final 3CX DesktopApp compromise was caused by an earlier X_TRADER software supply chain compromise, not merely an isolated bundled-library issue.",SRC-001SRC-002SRC-003
CONTRA-002,"Public reporting described a potentially broad compromise of 3CX software used by many enterprise customers, while 3CX and Kaspersky reported that follow-on payload execution was selective and that Gopuram was deployed to fewer than ten observed machines.","The compromised installer had broad distribution risk, but the actor appears to have filtered victims and deployed later-stage payloads selectively. Analysts should separate exposure scale from confirmed deeper compromise.",SRC-001SRC-004SRC-009


## Reasoning Chain

Step,Premise Entities,Premise Relations,Conclusion
1,ENT-001ENT-002,trojanized,"The investigation begins with a confirmed supply chain compromise of 3CX DesktopApp, affecting specific Windows and macOS Electron app versions."
2,ENT-003ENT-004ENT-005ENT-006,provided_softwaredistributed_as_installercontained_malware,The initial vector was upstream of 3CX: a Trading Technologies X_TRADER installer contained VEILEDSIGNAL malware.
3,ENT-006ENT-001ENT-013,enabled_initial_compromise_of_employee_system_leading_to_corporate_accessused_for_lateral_movement,The compromised X_TRADER installation enabled the actor to move from an employee system into the 3CX corporate environment.
4,ENT-001ENT-014ENT-015ENT-016,used_on_macos_build_serverused_on_windows_build_environment,"The actor compromised 3CX build infrastructure on both Windows and macOS, enabling the later trojanization of 3CX DesktopApp."
5,ENT-002ENT-010ENT-011ENT-019,executed_downloaderdownloaded_follow_on_payloadretrieved_c2_data_from_encrypted_icon_files_hosted_on,"The trojanized 3CX DesktopApp used a staged malware chain involving SUDDENICON, GitHub-hosted encrypted icon files, and ICONICSTEALER."
6,ENT-007ENT-008ENT-009ENT-012ENT-025,overlaps_with_north_korean_lazarus_related_activityuses_or_is_associated_withtargeted,"The public attribution points to UNC4736, a suspected North Korean-nexus cluster, with overlap to Lazarus/AppleJeus cryptocurrency-focused activity."


## Evaluation Notes

This case is intended to test multi-hop OSINT reasoning, not live intrusion analysis. Evaluators should reward systems that preserve source confidence levels, cite the exact source behind each attribution claim, distinguish exposure from confirmed follow-on compromise, and avoid collapsing UNC4736, Lazarus Group, and AppleJeus into a single unsupported identity. All URLs in sources were opened with the web tool on 2026-06-29 and included only if reachable with relevant content. Candidate sources that returned internal errors or could not be verified were discarded.